In [1]:
import os
import re

import numpy as np
import pandas as pd
# import torch
from tqdm.notebook import tqdm
from datetime import time, timedelta

In [2]:
pd.set_option('display.max_columns', None)

# functions defintion 

In [3]:
def read_gem_file(path:str, )-> pd.DataFrame: 
    colspecs = [(0, 5), (6, 13), (14, 19)]
    colnames = ["icd9", "icd10", "flags"]

    gems = pd.read_fwf(path, colspecs=colspecs, names=colnames, dtype=str)
    gems["icd9"] = gems["icd9"].str.strip()
    gems["icd10"] = gems["icd10"].str.strip()
    gems[["approximate", "no_map", "combination", "scenario", "choice_list"]] = \
        (gems["flags"].apply(lambda x: pd.Series(list(x))).astype(int))

    gems.drop(columns=["flags"], inplace=True)
    
    return gems

In [4]:
def clean_race(race):
    splits = race.split('//')
    
    if splits[1].startswith('WHITE'):
        splits[1] =  'WHITE'
        
    elif splits[1].startswith('UNABLE'):
        splits[1] =  'UNKNOWN'

    elif splits[1].startswith('BLACK'):
        splits[1] =  'BLACK'

    elif splits[1].startswith('PATIENT'):
        splits[1] =  'UNKNOWN'
        
    elif splits[1].startswith('ASIAN'):
        splits[1] =  'ASIAN'

    elif splits[1].startswith('HISPANIC'):
        splits[1] =  'HISPANIC'

    elif splits[1].startswith('NATIVE'):
        splits[1] =  'NATIVE HAWAIIAN'

    elif splits[1].startswith('AMERICAN INDIAN'):
        splits[1] =  'AMERICAN INDIAN'
        
    return '//'.join(splits)

In [5]:
def clean_outpatient_measurements(
    df: pd.DataFrame,
    pre_post_buffer: pd.Timedelta = timedelta(days=1),
    max_gap_days: int = 30,
    lab_like=("LAB", "MICROBIOLOGY"),
    admission_code="ADMISSION-AT-HOSPITAL",
    discharge_code="DISCHARGE-FROM-HOSPITAL",
) -> pd.DataFrame:
    """
    Update orphan LAB/MICROBIOLOGY events:
    - If within ±1 day of an admission: assign to `hadm_id` directly.
    - Else if within `max_gap_days`: assign to `out_id`.
    - Else: remove the event.

    Parameters:
    - df: Input DataFrame with ['subject_id', 'hadm_id', 'time', 'code']
    - pre_post_buffer: Time window to assign directly to hadm_id (ER events)
    - max_gap_days: Time window to link to out_id (historical context)
    - lab_like: Code prefixes considered lab/microbiology
    - admission_code / discharge_code: Strings identifying admission/discharge events

    Returns:
    - Filtered DataFrame with updated `hadm_id` and `out_id` columns
    """
    out = df.copy()
    out["time"] = pd.to_datetime(out["time"], errors="coerce")
    out["code_type"] = out["code"].str.split("//").str[0]
    out["out_id"] = np.nan
    out["__orig_order"] = range(len(out))

    visits = out[out["code"].str.contains(f"{admission_code}", na=False)].copy()
    visits = visits[["subject_id", "hadm_id", "time"]].dropna()
    visits = visits.sort_values(["subject_id", "time"])

    orphan_mask = out["hadm_id"].isna() & out["code_type"].isin(lab_like)
    orphan = out[orphan_mask].copy()
    rows_to_drop = []

    for idx, row in orphan.iterrows():
        sid, t_event = row["subject_id"], row["time"]
        patient_visits = visits[visits["subject_id"] == sid]

        if pd.isna(t_event) or patient_visits.empty:
            rows_to_drop.append(idx)
            continue

        patient_visits = patient_visits.copy()
        patient_visits["delta"] = (patient_visits["time"] - t_event).abs()
        nearest = patient_visits.loc[patient_visits["delta"].idxmin()]
        delta = nearest["delta"]

        if delta <= pre_post_buffer:
            out.at[idx, "hadm_id"] = nearest["hadm_id"]
        elif delta <= timedelta(days=max_gap_days):
            out.at[idx, "out_id"] = nearest["hadm_id"]
        else:
            rows_to_drop.append(idx)

    out = out.drop(index=rows_to_drop)
    out = out.sort_values("__orig_order").drop(columns="__orig_order").reset_index(drop=True)
    return out

In [6]:
def push_procedure_and_sort(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adjust PROCEDURE timestamps and properly order events per patient, visit-aware.
    
    Steps:
    - Push PROCEDURE timestamps at midnight to 23:59:59
    - Ensure RACE/GENDER always at the top of each patient's sequence
    - Sort events by:
        1. subject_id
        2. static (True first)
        3. earliest time per hadm_id (to respect admission order)
        4. event time within visits
    """
    df = df.copy()
    
    # Fix PROCEDURE timestamps at 00:00:00 → end of day
    mask_procedure = df["code"].str.startswith("PROCEDURE")
    mask_midnight = df["time"].dt.time == time(0, 0)
    df.loc[mask_procedure & mask_midnight, "time"] = (
        df.loc[mask_procedure & mask_midnight, "time"].dt.normalize() +
        pd.Timedelta(hours=23, minutes=59, seconds=59)
    )
    
    # Mark static tokens like RACE and GENDER
    df["is_static"] = df["code"].str.startswith(("RACE", "GENDER"))
    
    # Compute earliest time per hadm_id per patient to get admission order
    admission_order = (
        df[~df["hadm_id"].isna()]
        .groupby(["subject_id", "hadm_id"])["time"]
        .min()
        .reset_index()
        .rename(columns={"time": "hadm_order_time"})
    )
    
    df = df.merge(admission_order, on=["subject_id", "hadm_id"], how="left")

    # NaN hadm_id (outpatient) → set hadm_order_time = time (so it's positioned by actual time)
    df["hadm_order_time"] = df["hadm_order_time"].fillna(df["time"])

    # Final sort: static first, then visit order, then time
    df = df.sort_values(
        by=["subject_id", "is_static", "hadm_order_time", "time"],
        ascending=[True, False, True, True]
    ).drop(columns=["is_static", "hadm_order_time"]).reset_index(drop=True)

    return df

In [7]:
def build_best_icd10_mapper(gems_df: pd.DataFrame) -> dict:
    """
    Build best ICD-9 → ICD-10 mapping from GEMs file, prioritizing:
    - no_map == 0 (valid map)
    - combination == 0 (single-code map)
    - approximate == 0 (exact match preferred)
    - lowest choice_list (preferred target)

    Assumes ICD-9 codes are already in the same format as the target dataset.
    """
    df = gems_df.copy()

    # Filter for valid, standalone, mappable entries
    df = df[(df["no_map"] == 0) & (df["combination"] == 0)]

    # Ranking mechanism: prioritize exact match, low choice_list, shorter code
    df["rank"] = (
        df["approximate"] * 100 +
        df["choice_list"] * 10 +
        df["icd10"].str.len()
    )

    # Choose best ICD-10 per ICD-9
    best = (
        df.sort_values(["icd9", "rank"])
        .drop_duplicates(subset="icd9", keep="first")
    )

    # No padding applied to ICD-9 codes
    return dict(zip(best["icd9"].astype(str), best["icd10"].astype(str)))

In [8]:
mimic_data_path = os.path.join('..','data','raw','mimic-iv-meds','data','train')
mimic_metadata_path = os.path.join('..','data','raw','mimic-iv-meds','metadata',)
icd_cm_mapping_path = os.path.join('..','resources','icd-code-conversion','diagnosis','diagnosis_gems_2018','2018_I9gem.txt')
icd_pcs_mapping_path = os.path.join('..','resources','icd-code-conversion','procedure','procedure_gems_2018','gem_i9pcs.txt')

In [9]:
shard = pd.read_parquet(os.path.join(mimic_data_path,'0.parquet'))
gems_cm = read_gem_file(icd_cm_mapping_path)
gems_pcs = read_gem_file(icd_pcs_mapping_path)

In [10]:
# Patient without hospital admission
patients_without_hadm = []
for pid in tqdm(shard.subject_id.unique()):
    patient = shard[shard.subject_id == pid]
    patient_hadm_id = patient.hadm_id.unique()
    if patient_hadm_id.shape[0] == 1:
        patients_without_hadm.append(int(pid))

# filter
shard = shard[shard.subject_id.isin(patients_without_hadm) == False].reset_index(drop=True)    

  0%|          | 0/999 [00:00<?, ?it/s]

In [11]:
# remove table name from code
shard['table'] = shard.code.apply(lambda x: x.split('//')[-1])
shard.code = shard.code.apply(lambda x: '//'.join(x.split('//')[:-1]))

In [12]:
# handle patient race
# unify races
shard['race'] = shard.code.apply(lambda x: x.split('//')[1] if x.startswith('RACE') else np.nan)
shard['code'] = shard.code.apply(lambda x: clean_race(x) if x.startswith('RACE') else x)

# Step 1: Create base sequential 'filter' column
shard["filter"] = range(len(shard))

# Step 2: Identify RACE rows
race_mask = shard["code"].str.contains("RACE", na=False)

# Step 3: Assign the same filter value for consecutive RACE rows
filter_values = []
group_id = -1
for i, is_race in tqdm(enumerate(race_mask)):
    if i == 0 or not is_race or not race_mask.iloc[i - 1]:
        group_id += 1
    filter_values.append(group_id)
shard["filter"] = filter_values

# Step 4: Collapse duplicates by keeping first row per filter group
shard = shard.groupby("filter", as_index=False).first()
shard.drop(columns=['filter'],inplace=True)
patients_with_multiple_races = []
for pid in tqdm(shard.subject_id.unique()):
    patient = shard[shard.subject_id == pid]
    patient_races = patient[patient.code.str.startswith('RACE')]
    if patient_races.shape[0] > 2:
        patients_with_multiple_races.append(int(pid))
        
print(len(patients_with_multiple_races))

0it [00:00, ?it/s]

  0%|          | 0/615 [00:00<?, ?it/s]

0


In [13]:
# clean outpatient measuerments
cleaned_timelines = []
for pid in tqdm(shard.subject_id.unique()):
    patient = shard[shard.subject_id == pid]
    cleaned_patient = clean_outpatient_measurements(patient)
    cleaned_timelines.append(cleaned_patient)

shard = pd.concat(cleaned_timelines).reset_index(drop=True)
del(cleaned_timelines)

  0%|          | 0/615 [00:00<?, ?it/s]

In [14]:
shard.shape

(1510050, 56)

In [15]:
# clean empty admissions
empty_hadms = []
patients_with_empty_hadms = []

for pid in tqdm(shard.subject_id.unique()):
    patient = shard[shard.subject_id == pid]
    admissions = patient.hadm_id.dropna().unique()

    for hid in admissions:
        admission = patient[patient.hadm_id == hid].reset_index(drop=True)
        age_rows = admission[admission.code.str.startswith('AGE_AT_ADMISSION')]

        if not age_rows.empty:
            idx = age_rows.index[0]
            if idx + 1 < len(admission):
                next_code = admission.iloc[idx + 1].code
                if next_code.startswith('DISCHARGE-FROM-HOSPITAL'):
                    empty_hadms.append(int(hid))
                    patients_with_empty_hadms.append(int(pid))

patients_with_single_empty = []
for pid in patients_with_empty_hadms:
    patient = shard[shard.subject_id == pid]
    hids = patient.hadm_id.unique()
#     print(hids.shape[0],pid)
    if hids.shape[0] == 2:
        patients_with_single_empty.append(pid)
        
patients_with_empty_hadms = [pid for pid in patients_with_empty_hadms if \
                             pid not in patients_with_single_empty]

shard = shard[shard.subject_id.isin(patients_with_single_empty) == False].reset_index(drop=True)

shard = shard[(shard.hadm_id.isin(empty_hadms) == False) & 
              (shard.out_id.isin(empty_hadms) == False)].reset_index(drop=True)

  0%|          | 0/615 [00:00<?, ?it/s]

In [16]:
# handle procedure codes
shard = push_procedure_and_sort(shard)

In [17]:
# handle diagnosis codes mapping
gems_cm = gems_cm[gems_cm.no_map != 1]
icd9_to_icd10_cm = build_best_icd10_mapper(gems_cm)

# map exact approximate
shard['icd9_to_icd10'] = shard.diag_icd_code.map(icd9_to_icd10_cm)

# map combinations
gems_cm_comb = gems_cm[(gems_cm.combination == 1)& (gems_cm.scenario == 1)].iloc[:,:2]
shard = shard.merge(gems_cm_comb,left_on='diag_icd_code',right_on='icd9',how='left')
shard.icd9_to_icd10 = shard.apply(lambda x: x['icd10'] if pd.notna(x['icd10']) else x['icd9_to_icd10'], axis=1)

# remove unmatched
all_icd9 = shard[shard.diag_version == 9]
unmapped_icd9 = all_icd9[all_icd9.icd9_to_icd10.isna()].code.unique()
shard = shard[shard.code.isin(unmapped_icd9) == False].reset_index(drop=True)
# unify names
shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.icd9_to_icd10]) if x.code.startswith('DIAGNOSIS-ICD//9') else x.code,axis=1)
shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.diag_icd_code]) if x.code.startswith('DIAGNOSIS-ICD//10') else x.code,axis=1)

shard.drop(columns=['icd9','icd10'],inplace=True)

In [18]:
shard[shard.code.str.startswith('PROC')].shape

(2278, 57)

In [19]:
gems_pcs = gems_pcs[gems_pcs.no_map != 1]
icd9_to_icd10_pcs = build_best_icd10_mapper(gems_pcs)

# map exact approximate
shard['icd9_to_icd10'] = shard.proc_icd_code.map(icd9_to_icd10_pcs)

# map combinations
gems_pcs_comb = gems_pcs[(gems_pcs.combination == 1)& (gems_pcs.scenario == 1)].iloc[:,:2]
shard = shard.merge(gems_pcs_comb,left_on='proc_icd_code',right_on='icd9',how='left')
shard.icd9_to_icd10 = shard.apply(lambda x: x['icd10'] if pd.notna(x['icd10']) else x['icd9_to_icd10'], axis=1)

# remove unmatched
all_icd9 = shard[shard.proc_version == 9]
unmapped_icd9 = all_icd9[all_icd9.icd9_to_icd10.isna()].code.unique()
shard = shard[shard.code.isin(unmapped_icd9) == False].reset_index(drop=True)

# unify names
shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.icd9_to_icd10]) if x.code.startswith('PROCEDURE-ICD//9') else x.code,axis=1)
shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.proc_icd_code]) if x.code.startswith('PROCEDURE-ICD//10') else x.code,axis=1)

shard.drop(columns=['icd9','icd10'],inplace=True)

In [20]:
shard

,subject_id,time,code,numeric_value,admission_type,admission_location,hadm_id,discharge_location,died_in_hosp,diag_version,diag_icd_code,diag_seq_num,drg_severity,drg_mortality,drg_type,drg_code,text_value,priority,specimen_id,lab_lower_limit,lab_upper_limit,lab_flag,lab_unit,lab_itemid,gender,route,frequency,doses_per_24_hrs,medication,proc_seq_num,proc_version,proc_icd_code,micro_specimen_id,micro_org_name,micro_test_name,micro_spec_type_desc,micro_test_itemid,icustay_id,icu_care_unit,icu_los,category,label,itemid,abbreviation,rate,unit,amount,amountuom,ordercategorydescription,ordercategoryname,secondaryordercategoryname,ordercomponenttypedescription,table,race,code_type,out_id,icd9_to_icd10
0,10005866,NaT,RACE//PORTUGUESE,NaN,None,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/admissions,PORTUGUESE,RACE,NaN,NaN
1,10005866,NaT,GENDER//M,NaN,None,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,M,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/patients,None,GENDER,NaN,NaN
2,10005866,2146-06-05 22:40:00,LAB//51237,1.4,None,None,26158160.0,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,None,STAT,20940795.0,0.9,1.1,abnormal,None,51237.0,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/labevents,None,LAB,NaN,NaN
3,10005866,2146-06-05 22:40:00,LAB//51274,15.3,None,None,26158160.0,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,None,STAT,20940795.0,9.4,12.5,abnormal,sec,51274.0,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/labevents,None,LAB,NaN,NaN
4,10005866,2146-06-05 22:40:00,LAB//51275,34.5,None,None,26158160.0,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,None,STAT,20940795.0,25.0,36.5,None,sec,51275.0,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/labevents,None,LAB,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1502940,19978119,2189-04-28 17:23:00,DIAGNOSIS-ICD//H269,NaN,None,None,20178379.0,None,NaN,10.0,H269,24.0,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/diagnoses_icd,None,DIAGNOSIS-ICD,NaN,NaN
1502941,19978119,2189-04-28 17:23:00,DIAGNOSIS-ICD//Z87891,NaN,None,None,20178379.0,None,NaN,10.0,Z87891,25.0,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/diagnoses_icd,None,DIAGNOSIS-ICD,NaN,NaN
1502942,19978119,2189-04-28 17:23:00,DIAGNOSIS-ICD//Z66,NaN,None,None,20178379.0,None,NaN,10.0,Z66,26.0,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/diagnoses_icd,None,DIAGNOSIS-ICD,NaN,NaN
1502943,19978119,2189-04-28 17:23:00,DRG//APR//720,NaN,None,None,20178379.0,None,NaN,NaN,None,NaN,4.0,4.0,APR,720.0,None,None,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/drgcodes,None,DRG,NaN,NaN


In [21]:
shard.groupby(['subject_id','hadm_id']).count()#.sort_values(by='code').iloc[500:]

time  code  numeric_value  admission_type  \
subject_id hadm_id                                                 
10005866   20364112.0  4355  4355           2428               1   
           21636229.0   249   249            203               1   
           22589518.0   222   222            164               1   
           23514107.0   234   234            192               1   
           26134779.0   420   420            342               1   
...                     ...   ...            ...             ...   
19960537   25246150.0   156   156            124               1   
19978119   20178379.0  2195  2195           1048               1   
           21747256.0    86    86             44               1   
           24233127.0   275   275            173               1   
           24540502.0   240   240            172               1   

                       admission_location  discharge_location  died_in_hosp  \
subject_id hadm_id                                                            
10005866   20364112.0                   1                   2             1   
           21636229.0                   1                   2             1   
           22589518.0                   1                   2             1   
           23514107.0                   1                   2             1   
           26134779.0                   1                   2             1   
...                                   ...                 ...           ...   
19960537   25246150.0                   1                   2             1   
19978119   20178379.0                   1                   2             1   
           21747256.0                   1                   0             1   
           24233127.0                   1                   2             1   
           24540502.0                   1                   2             1   

                       diag_version  diag_icd_code  diag_seq_num  \
subject_id hadm_id                                                 
10005866   20364112.0            28             28            28   
           21636229.0             7              7             7   
           22589518.0             9              9             9   
           23514107.0             9              9             9   
           26134779.0             6              6             6   
...                             ...            ...           ...   
19960537   25246150.0            10             10            10   
19978119   20178379.0            26             26            26   
           21747256.0            14             14            14   
           24233127.0            22             22            22   
           24540502.0            20             20            20   

                       drg_severity  drg_mortality  drg_type  drg_code  \
subject_id hadm_id                                                       
10005866   20364112.0             1              1         2         2   
           21636229.0             1              1         2         2   
           22589518.0             1              1         2         2   
           23514107.0             1              1         2         2   
           26134779.0             1              1         2         2   
...                             ...            ...       ...       ...   
19960537   25246150.0             1              1         2         2   
19978119   20178379.0             1              1         2         2   
           21747256.0             0              0         0         0   
           24233127.0             1              1         2         2   
           24540502.0             1              1         2         2   

                       text_value  priority  specimen_id  lab_lower_limit  \
subject_id hadm_id                                                          
10005866   20364112.0        2058       991         1060              917   
           21636229.0          

In [42]:
shard[shard.hadm_id ==28640981 ]

,subject_id,time,code,numeric_value,admission_type,admission_location,hadm_id,discharge_location,died_in_hosp,diag_version,diag_icd_code,diag_seq_num,drg_severity,drg_mortality,drg_type,drg_code,text_value,priority,specimen_id,lab_lower_limit,lab_upper_limit,lab_flag,lab_unit,lab_itemid,gender,route,frequency,doses_per_24_hrs,medication,proc_seq_num,proc_version,proc_icd_code,micro_specimen_id,micro_org_name,micro_test_name,micro_spec_type_desc,micro_test_itemid,icustay_id,icu_care_unit,icu_los,category,label,itemid,abbreviation,rate,unit,amount,amountuom,ordercategorydescription,ordercategoryname,secondaryordercategoryname,ordercomponenttypedescription,table,race,code_type,out_id,icd9_to_icd10
331552,11696830,2161-02-10 20:30:00,LAB//51237,1.1,None,None,28640981.0,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,None,STAT,1646304.0,0.9,1.1,None,None,51237.0,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/labevents,None,LAB,NaN,NaN
331553,11696830,2161-02-10 20:30:00,LAB//51274,11.9,None,None,28640981.0,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,None,STAT,1646304.0,9.4,12.5,None,sec,51274.0,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/labevents,None,LAB,NaN,NaN
331554,11696830,2161-02-10 20:30:00,LAB//51275,29.0,None,None,28640981.0,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,None,STAT,1646304.0,25.0,36.5,None,sec,51275.0,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/labevents,None,LAB,NaN,NaN
331555,11696830,2161-02-10 20:30:00,LAB//50887,NaN,None,None,28640981.0,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,HOLD. DISCARD GREATER THAN 24 HRS OLD.,STAT,10010098.0,NaN,NaN,None,None,50887.0,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/labevents,None,LAB,NaN,NaN
331556,11696830,2161-02-10 20:30:00,LAB//50868,11.0,None,None,28640981.0,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,___,STAT,33133357.0,10.0,18.0,None,mEq/L,50868.0,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/labevents,None,LAB,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331676,11696830,2161-02-12 17:17:00,DIAGNOSIS-ICD//M48061,NaN,None,None,28640981.0,None,NaN,10.0,M48061,25.0,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/diagnoses_icd,None,DIAGNOSIS-ICD,NaN,NaN
331677,11696830,2161-02-12 17:17:00,DIAGNOSIS-ICD//M47892,NaN,None,None,28640981.0,None,NaN,10.0,M47892,26.0,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/diagnoses_icd,None,DIAGNOSIS-ICD,NaN,NaN
331678,11696830,2161-02-12 17:17:00,DIAGNOSIS-ICD//K219,NaN,None,None,28640981.0,None,NaN,10.0,K219,27.0,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/diagnoses_icd,None,DIAGNOSIS-ICD,NaN,NaN
331679,11696830,2161-02-12 17:17:00,DIAGNOSIS-ICD//G629,NaN,None,None,28640981.0,None,NaN,10.0,G629,28.0,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,NaN,None,None,None,NaN,None,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,None,None,NaN,None,NaN,None,NaN,None,None,None,None,None,hosp/diagnoses_icd,None,DIAGNOSIS-ICD,NaN,NaN


In [106]:
pd.read_csv('/scratch/fs999/shamoutlab/data/ccad_emr/ccad_stroke_May2021/DiagnosisEventFact.csv',encoding='windows-1252')

,PatientDurableKey,PatientKey,EncounterKey,EncounterInstant,DiagnosisEventKey,DiagnosisEventStartDate,DiagnosisEventEndDate,DiagnosisKey,DiagnosisCodeSet,DiagnosisCode,DiagnosisName,Diagnosis_ICD10_Name_From_CCSR_File,Diagnosis_CCSR,Diagnosis_CCSR_Cat_Description,DepartmentKey,DepartmentName,DiagnosisEventType,DiagnosisEventStatus,DiagnosisEventPresentOnAdmission
0,1,488338,5545801,2015-07-22T21:00:00Z,13888060,NaN,NaN,1236577,ICD-10-CM,F419,Anxiety,"Anxiety disorder, unspecified",MBD005,Anxiety and fear-related disorders,2320,COLORECTAL SURG CAD,Medical History,Active,*Not Applicable
1,1,1,6645750,2015-07-22T21:00:00Z,10854989,2015-07-23,2015-07-23,165038,ICD-10-CM,R1013,Epigastric pain,Epigastric pain,SYM006,Abdominal pain and other digestive/abdomen sig...,2315,AMB LAB CAD,Encounter Diagnosis,Active,*Not Applicable
2,1,1,5545801,2015-07-22T21:00:00Z,10854253,2015-07-23,2015-07-23,540939,ICD-10-CM,K589,IBS (irritable bowel syndrome),Irritable bowel syndrome without diarrhea,DIG025,Other specified and unspecified gastrointestin...,2320,COLORECTAL SURG CAD,Encounter Diagnosis,Active,*Not Applicable
3,1,1,5117112,2015-07-22T21:00:00Z,10854772,2015-07-23,2015-07-23,165038,ICD-10-CM,R1013,Epigastric pain,Epigastric pain,SYM006,Abdominal pain and other digestive/abdomen sig...,2324,GASTROENTEROLOGY CAD,Encounter Diagnosis,Active,*Not Applicable
4,1,1,4396763,2015-07-26T21:00:00Z,10857483,2015-07-27,2015-07-27,165038,ICD-10-CM,R1013,Epigastric pain,Epigastric pain,SYM006,Abdominal pain and other digestive/abdomen sig...,2324,GASTROENTEROLOGY CAD,Encounter Diagnosis,Active,*Not Applicable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9974071,654671,654671,10997710,2021-05-25T18:52:44Z,21470718,2021-05-25,2021-05-25,748752,ICD-10-CM,M62838,Muscle spasm,Other muscle spasm,MUS026,Muscle disorders,2404,EMERGENCY HAD,Encounter Diagnosis,Active,*Not Applicable
9974072,654673,654673,10997738,2021-05-25T18:51:51Z,21484001,NaN,NaN,1245308,ICD-10-CM,I959,Hypotension,"Hypotension, unspecified",CIR031,Hypotension,2404,EMERGENCY HAD,Medical History,Active,*Not Applicable
9974073,654673,654673,10997738,2021-05-25T18:51:51Z,21470724,2021-05-25,2021-05-25,1084381,ICD-10-CM,R55,Near syncope,Syncope and collapse,SYM001,Syncope,2404,EMERGENCY HAD,Encounter Diagnosis,Active,*Not Applicable
9974074,654675,654675,10997746,2021-05-25T18:46:10Z,21484002,NaN,NaN,637425,ICD-10-CM,E119,DM (diabetes mellitus),Type 2 diabetes mellitus without complications,END002,Diabetes mellitus without complication,2404,EMERGENCY HAD,Medical History,Active,*Not Applicable
